In [1]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False



Welcome to JupyROOT 6.18/00


In [2]:
def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY_jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W_jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z_jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi1995_Mchi1000"] = ("Mphi-1995_Mchi-1000")
    sig_map["Mphi2000_Mchi1500"] = ("Mphi-2000_Mchi-1500")
    sig_map["Mphi2000_Mchi500"] = ("Mphi-2000_Mchi-500")
#     haven't processed signal yet
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mchi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists, "sig":signal_hists}

    return hists


In [ ]:
np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])

In [5]:
year = '2016'

mT_corrected_binning_dict = {
    "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
}
mT_corrected_binning = mT_corrected_binning_dict[year]
mT_correctedbins = np.array(mT_corrected_binning)
nmT_corrected = len(mT_correctedbins) - 1

###
# Extract histograms from input file
###

print("Grouping histograms")
hists = load("../../../hists/UL_lep_monotop_v1_test_preVFP_" + year + ".scaled")
hists = remap_histograms(hists)
data_hists = hists["data"]
bkg_hists = hists["bkg"]
signal_hists = hists["sig"]

###
# Preparing histograms for fit
##

data = {}
for r in data_hists["template"].identifiers("region"):
    data[str(r)] = data_hists["template"].integrate("region", r)

background = {}
for r in bkg_hists["template"].identifiers("region"):
    background[str(r)] = bkg_hists["template"].integrate("region", r)

signal = {}
for r in signal_hists["template"].identifiers("region"):
    signal[str(r)] = signal_hists["template"].integrate("region", r)
    
    

Grouping histograms
initial mT_corrected binning [100. 125. 150. 165. 180. 200. 220. 240. 270. 310. 350. 500.]


In [ ]:
addVJetsSyst(background,"DY_jets", "ttbare", ttbare_dyjets)

In [15]:
background['wjete'].identifiers("systematic")

[<StringBin (btagDown) instance at 0x7f3d7da2b080>,
 <StringBin (btagUp) instance at 0x7f3d7da2b048>,
 <StringBin (ew1Down) instance at 0x7f3d7da2b978>,
 <StringBin (ew1Up) instance at 0x7f3d7da2b908>,
 <StringBin (ew2GDown) instance at 0x7f3d7da2ba20>,
 <StringBin (ew2GUp) instance at 0x7f3d7da2b9e8>,
 <StringBin (ew2WDown) instance at 0x7f3d7da2bac8>,
 <StringBin (ew2WUp) instance at 0x7f3d7da2ba90>,
 <StringBin (ew2ZDown) instance at 0x7f3d7da2bb70>,
 <StringBin (ew2ZUp) instance at 0x7f3d7da2bb38>,
 <StringBin (ew3GDown) instance at 0x7f3d7da2bc18>,
 <StringBin (ew3GUp) instance at 0x7f3d7da2bbe0>,
 <StringBin (ew3WDown) instance at 0x7f3d7da2bcc0>,
 <StringBin (ew3WUp) instance at 0x7f3d7da2bc88>,
 <StringBin (ew3ZDown) instance at 0x7f3d7da2bd68>,
 <StringBin (ew3ZUp) instance at 0x7f3d7da2bd30>,
 <StringBin (id_eDown) instance at 0x7f3d7da2b198>,
 <StringBin (id_eUp) instance at 0x7f3d7da2b160>,
 <StringBin (isolation_eDown) instance at 0x7f3d7da2b278>,
 <StringBin (isolation_eU

In [11]:
def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
#     print("btagUp", btagUp)
#     print("btagDo", btagDown)
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
#         print(string)
        histogram = dictionary[region].integrate("process", process)
#         print(histogram.integrate("systematic", "nominal").sum('npvs').values()[()])
        nominal=histogram.integrate("systematic", "nominal").sum('npvs').values()[()]
        up=histogram.integrate("systematic", string+"Up").sum('npvs').values()[()]
        down=histogram.integrate("systematic",string+"Down").sum('npvs').values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
#         print(systUp)
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
    addSyst(dictionary, process, region, templ, ew2W, "ew2W")
    addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
    addSyst(dictionary, process, region, templ, ew3W, "ew3W")
    addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
    addSyst(dictionary, process, region, templ, mix, "mix")
    addSyst(dictionary, process, region, templ, qcd1, "qcd1")
    addSyst(dictionary, process, region, templ, qcd2, "qcd2")
    addSyst(dictionary, process, region, templ, qcd3, "qcd3")
    addSyst(dictionary, process, region, templ, muR, "muR")
    addSyst(dictionary, process, region, templ, muF, "muF")

In [23]:
histogram = background['wjete'].integrate("process", "W_jets")
histogram.integrate("systematic", "ew1Up").sum('npvs')

<Hist (mT_corrected) instance at 0x7f3d6b8f8b70>

In [25]:
histogram.identifiers('systematic')

[<StringBin (btagDown) instance at 0x7f3d7da2b080>,
 <StringBin (btagUp) instance at 0x7f3d7da2b048>,
 <StringBin (id_eDown) instance at 0x7f3d7da2b198>,
 <StringBin (id_eUp) instance at 0x7f3d7da2b160>,
 <StringBin (isolation_eDown) instance at 0x7f3d7da2b278>,
 <StringBin (isolation_eUp) instance at 0x7f3d7da2b240>,
 <StringBin (nominal) instance at 0x7f3d7da3cf98>,
 <StringBin (pileupDown) instance at 0x7f3d7da2b2e8>,
 <StringBin (pileupUp) instance at 0x7f3d7da2b2b0>,
 <StringBin (reco_eDown) instance at 0x7f3d7da2b208>,
 <StringBin (reco_eUp) instance at 0x7f3d7da2b1d0>,
 <StringBin (trig_eDown) instance at 0x7f3d7da2b0f0>,
 <StringBin (trig_eUp) instance at 0x7f3d7da2b0b8>]

In [7]:
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process).sum('npvs')
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT_corrected")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT_corrected", sumw2)
    return (output, binning, "mT_corrected")

In [8]:
###
###
# Setting up systematics
###
###
lumi = rl.NuisanceParameter("lumi" + year, "lnN")
# zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
# wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
# tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
id_e = rl.NuisanceParameter("id_e" + year, "shape")
id_m = rl.NuisanceParameter("id_m" + year, "shape")
# id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
# reco_e = rl.NuisanceParameter("reco_e" + year, "lnN")
# iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
trig_e = rl.NuisanceParameter("trig_e" + year, "shape")
trig_m = rl.NuisanceParameter("trig_m" + year, "shape")
# trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
# trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
# veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
# jec = rl.NuisanceParameter("jec" + year, "lnN")
# btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
ew1 = rl.NuisanceParameter("ew1", "shape")
# #ew2G = rl.NuisanceParameter("ew2G", "lnN")
ew2W = rl.NuisanceParameter("ew2W", "shape")
ew2Z = rl.NuisanceParameter("ew2Z", "shape")
# #ew3G = rl.NuisanceParameter("ew3G", "lnN")
ew3W = rl.NuisanceParameter("ew3W", "shape")
ew3Z = rl.NuisanceParameter("ew3Z", "shape")
mix = rl.NuisanceParameter("mix", "shape")
muF = rl.NuisanceParameter("muF", "lnN")
muR = rl.NuisanceParameter("muR", "lnN")
qcd1 = rl.NuisanceParameter("qcd1", "shape")
qcd2 = rl.NuisanceParameter("qcd2", "shape")
qcd3 = rl.NuisanceParameter("qcd3", "shape")
# whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
# zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
#ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")
pileup = rl.NuisanceParameter("pileup"+year, "shapeN")

# ttbare

In [14]:
year='2016'
model_id = 'ControlRegionModel'
model = rl.Model(model_id)

ch_name = "ttbare" 
# Create a channel/Control region
ttbare = rl.Channel(ch_name)
# add the Channel/CR to the model
model.addChannel(ttbare)

###
# Add data distribution to the channel
###

if year == "2018":
    ttbare.setObservation(template(data, "EGamma", "data", "ttbare"))
else:
    ttbare.setObservation(template(data, "SingleElectron", "data","ttbare"))

###
# W(->lnu)+jets data-driven model
###

ttbare_wjetsTemplate = template(background, "W_jets", "nominal", "ttbare", read_sumw2=True)
ttbare_wjetsMC = rl.TemplateSample(
    ch_name + "_wjetsMC",
    rl.Sample.BACKGROUND,
    ttbare_wjetsTemplate
)
ttbare_wjetsMC.setParamEffect(lumi, 1.027)
# ttbare_wjetsMC.setParamEffect(trig_e, 1.01)
# ttbare_wjetsMC.setParamEffect(veto_tau, 1.03)
# ttbare_wjetsMC.setParamEffect(wjets_norm, 1.4)
# ttbare_wjetsMC.setParamEffect(jec, 1.05)
# ttbare_wjetsMC.setParamEffect(id_e, 1.02)
# ttbare_wjetsMC.setParamEffect(reco_e, 1.02)
# ttbare_wjetsMC.autoMCStats()
addBtagSyst(background, "W_jets", "ttbare", ttbare_wjetsMC)
#     #id Electron
id_eUp=template(background, "W_jets", 'id_eUp', 'ttbare')[0] 
id_eDown=template(background, "W_jets", 'id_eDown', 'ttbare')[0]
ttbare_wjetsMC.setParamEffect(id_e, id_eUp, id_eDown)
#print("id_eUP",id_eUp)
#print("id_eDown",id_eDown)
addVJetsSyst(background, "W_jets", "ttbare", ttbare_wjetsMC)
pileupUp=template(background, 'W_jets','pileupUp','ttbare')[0] 
pileupDown=template(background, 'W_jets','pileupDown','ttbare')[0] 


ttbare_wjetsMC.setParamEffect(pileup, pileupUp, pileupDown)

ttbare.addSample(ttbare_wjetsMC)

# ###
# # top-antitop model
# ###

ttbare_ttTemplate = template(background, "TT", "nominal","ttbare", read_sumw2=True)
ttbare_ttMC = rl.TemplateSample(
    ch_name + "_ttMC",
    rl.Sample.BACKGROUND,
    ttbare_ttTemplate
)
ttbare_ttMC.setParamEffect(lumi, 1.027)
# ttbare_ttMC.setParamEffect(trig_e, 1.01)
# ttbare_ttMC.setParamEffect(veto_tau, 1.03)
# ttbare_ttMC.setParamEffect(jec, 1.05)
# ttbare_ttMC.setParamEffect(id_e, 1.02)
# ttbare_ttMC.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "TT", "ttbare", ttbare_ttMC)
# ttbare_ttMC.setParamEffect(ttMC_norm, 1.2)
ttbare.addSample(ttbare_ttMC)

# ###
# # Other MC-driven processes
# ###
# ST Background
ttbare_stTemplate = template(background, "ST", "nominal", "ttbare")
ttbare_st = rl.TemplateSample(
    ch_name + "_stMC", rl.Sample.BACKGROUND, ttbare_stTemplate
)
ttbare_st.setParamEffect(lumi, 1.027)
# ttbare_st.setParamEffect()
# ttbare_st.setParamEffect(trig_e, 1.01)
# ttbare_st.setParamEffect(veto_tau, 1.03)
# # ttbare_st.setParamEffect(st_norm, 1.2) something related to signal
# ttbare_st.setParamEffect(jec, 1.05)
# ttbare_st.setParamEffect(id_e, 1.02)
# ttbare_st.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "ST", "ttbare", ttbare_st)
trigeUp=template(background, 'ST', 'trig_eUp', 'ttbare')[0] 
trigeDown=template(background, 'ST', 'trig_eDown', 'ttbare')[0]

ttbare_st.setParamEffect(trig_e, trigeUp, trigeDown)
ttbare.addSample(ttbare_st)

# DY Jet Background
ttbare_dyjetsTemplate = template(background, "DY_jets", "nominal","ttbare")
ttbare_dyjets = rl.TemplateSample(
    ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbare_dyjetsTemplate
)
ttbare_dyjets.setParamEffect(lumi, 1.027)
# ttbare_dyjets.setParamEffect(trig_e, 1.01)
# ttbare_dyjets.setParamEffect(veto_tau, 1.03)
# # ttbare_dyjets.setParamEffect(zjetsMC_norm, 1.4)
# ttbare_dyjets.setParamEffect(jec, 1.05)
# ttbare_dyjets.setParamEffect(id_e, 1.02)
# ttbare_dyjets.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "DY_jets", "ttbare", ttbare_dyjets)
addVJetsSyst(background,"DY_jets", "ttbare", ttbare_dyjets)
#print("hi")
#print(ttbare_dyjets.getParamEffect(ew1))
pileupUp=template(background, 'DY_jets','pileupUp','ttbare')[0] 
pileupDown=template(background, 'DY_jets','pileupDown','ttbare')[0] 

print("pileupUp", pileupUp)
print("pileupDown", pileupDown)
ttbare_dyjets.setParamEffect(pileup, pileupUp, pileupDown)
ttbare.addSample(ttbare_dyjets)


# V Jets Background
ttbare_vvTemplate = template(background, "VV", "nominal", "ttbare")
ttbare_vv = rl.TemplateSample(
    ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbare_vvTemplate
)
ttbare_vv.setParamEffect(lumi, 1.027)
# ttbare_vv.setParamEffect(trig_e, 1.01)
# ttbare_vv.setParamEffect(veto_tau, 1.03)
# # ttbare_vv.setParamEffect(vv_norm, 1.2)
# ttbare_vv.setParamEffect(jec, 1.05)
# ttbare_vv.setParamEffect(id_e, 1.02)
# ttbare_vv.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "VV", "ttbare", ttbare_vv)
ttbare.addSample(ttbare_vv)

# QCD background
ttbare_qcdTemplate = template(background, "QCD", "nominal", "ttbare")
ttbare_qcd = rl.TemplateSample(
    ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbare_qcdTemplate
)
ttbare_qcd.setParamEffect(lumi, 1.027)
# ttbare_qcd.setParamEffect(trig_e, 1.01)
# ttbare_qcd.setParamEffect(veto_tau, 1.03)
# # ttbare_qcd.setParamEffect(qcde_norm, 2.0)
# ttbare_qcd.setParamEffect(jec, 1.05)
# ttbare_qcd.setParamEffect(id_e, 1.02)
# ttbare_qcd.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "QCD", "ttbare", ttbare_qcd)

#id Electron
id_eUp=template(background, "QCD", 'id_eUp', 'ttbare')[0] 
id_eDown=template(background, "QCD", 'id_eDown', 'ttbare')[0]
ttbare_qcd.setParamEffect(id_e, id_eUp, id_eDown)
#print("id_eUp", id_eUp)
#print("id_eDown", id_eDown)
ttbare.addSample(ttbare_qcd)


for s in signal['ttbare'].identifiers('process'):
    #print(s)
    if 'Mphi1995_Mchi1000' not in str(s): continue
    ttbare_signalTemplate = template(signal,s,'nominal','ttbare')
    ttbare_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, ttbare_signalTemplate)
    ttbare_signal.setParamEffect(lumi, 1.027)
#     sr_signal.setParamEffect(trig_met, 1.01)
#     sr_signal.setParamEffect(veto_tau, 1.03)
#     sr_signal.setParamEffect(jec, 1.05)
    btagUp=template(signal, 'Mphi1995_Mchi1000','btagUp','ttbare')[0] 
    btagDown=template(signal, 'Mphi1995_Mchi1000','btagDown','ttbare')[0] 
#     #print("btag")
#     #print(btagDown)
    ttbare_signal.setParamEffect(btag, btagUp, btagDown)
#     #print(ttbare_signal.getParamEffect(btag))
    trigeUp=template(signal, 'Mphi1995_Mchi1000', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(signal, 'Mphi1995_Mchi1000', 'trig_eDown', 'ttbare')[0]
    ttbare_signal.setParamEffect(trig_e, trigeUp, trigeDown)
    
    recoeUp=template(signal, 'Mphi1995_Mchi1000', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(signal, 'Mphi1995_Mchi1000', 'reco_eDown', 'ttbare')[0]
    ttbare_signal.setParamEffect(trig_e, recoeUp, recoDown)

#     #print("reco")
#     #print(recoeUp)
    ttbare.addSample(ttbare_signal)
    
    
    

pileupUp [ 1.          1.          1.67096251  1.         10.36439332  0.50004183
  0.94870979  1.29406601  1.45682436  0.569206    1.7620847 ]
pileupDown [1.         1.         1.07346292 1.         2.60997529 0.77087546
 1.07563102 0.98968318 1.07216618 0.87876252 1.12847983]
effect_up (pileup2016, ttbare_dyjetsMC) has magnitude greater than 50% (73.22%), you might be passing absolute values instead of relative


In [ ]:
template(signal, 'Mphi1995_Mchi1000', 'trig_eUp', 'wjete')

In [35]:
signal['wjete'].integrate("process", "Mphi1995_Mchi1000").sum('npvs').integrate("systematic", 'trig_eUp').values()

{(): array([ 7.53661369, 16.23366055, 14.70197397, 13.23989184, 17.36249656,
        15.88282215, 15.97061372, 23.85615068, 32.1988018 , 21.49431074,
        55.94499508])}

In [10]:
model.name


'ControlRegionModel'

In [21]:
model.renderCombine("test_combine")



In [134]:
signal

{'sre': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9978>,
 'srm': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c99e8>,
 'ttbare': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9a20>,
 'ttbarm': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9a58>,
 'wjete': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9a90>,
 'wjetm': <Hist (process,systematic,npvs,mT_corrected) instance at 0x7fcc352c9ac8>}

In [38]:
background['ttbarm'].identifiers('systematic')

[<StringBin (btagDown) instance at 0x7fc519a78da0>,
 <StringBin (btagUp) instance at 0x7fc519a78d68>,
 <StringBin (ew1Down) instance at 0x7fc519a736d8>,
 <StringBin (ew1Up) instance at 0x7fc519a73668>,
 <StringBin (ew2GDown) instance at 0x7fc519a73780>,
 <StringBin (ew2GUp) instance at 0x7fc519a73748>,
 <StringBin (ew2WDown) instance at 0x7fc519a73828>,
 <StringBin (ew2WUp) instance at 0x7fc519a737f0>,
 <StringBin (ew2ZDown) instance at 0x7fc519a738d0>,
 <StringBin (ew2ZUp) instance at 0x7fc519a73898>,
 <StringBin (ew3GDown) instance at 0x7fc519a73978>,
 <StringBin (ew3GUp) instance at 0x7fc519a73940>,
 <StringBin (ew3WDown) instance at 0x7fc519a73a20>,
 <StringBin (ew3WUp) instance at 0x7fc519a739e8>,
 <StringBin (ew3ZDown) instance at 0x7fc519a73ac8>,
 <StringBin (ew3ZUp) instance at 0x7fc519a73a90>,
 <StringBin (id_mDown) instance at 0x7fc519a73160>,
 <StringBin (id_mUp) instance at 0x7fc519a73128>,
 <StringBin (isolation_mDown) instance at 0x7fc519a73240>,
 <StringBin (isolation_mU

In [113]:
for s in signal['wjete'].identifiers('process'):
    print(s)
    if 'Mhs_50' not in str(s): continue
    wjete_signalTemplate = template(signal,s,'nominal','wjete')
    wjete_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, wjete_signalTemplate)
    wjete_signal.setParamEffect(lumi, 1.027)
#     sr_signal.setParamEffect(trig_met, 1.01)
#     sr_signal.setParamEffect(veto_tau, 1.03)
#     sr_signal.setParamEffect(jec, 1.05)
    btagUp=template(signal, s,'btagUp','wjete')[0]
    btagDown=template(signal, s,'btagDown','wjete')[0]
    wjete_signal.setParamEffect(btag, btagUp, btagDown)
    wjete.addSample(wjete_signal)
    


Mphi1995_Mchi1000
Mphi2000_Mchi1500
Mphi2000_Mchi500


In [114]:
def renderRoofit(self, workspace):
    import ROOT
    install_roofit_helpers()
    pdfName = self.name + '_simPdf'
    dataName = self.name + '_observation'
    rooSimul = workspace.pdf(pdfName)
    rooData = workspace.data(dataName)
    if rooSimul == None and rooData == None:  # noqa: E711
        channelCat = ROOT.RooCategory(self.name + '_channel', self.name + '_channel')
        # TODO s+b, b-only separate?
        rooSimul = ROOT.RooSimultaneous(self.name + '_simPdf', self.name + '_simPdf', channelCat)
        obsmap = ROOT.std.map('string, RooDataHist*')()
        for channel in self:
            channelCat.defineType(channel.name)
            pdf, obs = channel.renderRoofit(workspace)
            rooSimul.addPdf(pdf, channel.name)
            # const string magic: https://root.cern.ch/phpBB3/viewtopic.php?f=15&t=16882&start=15#p86985
            obsmap.insert(ROOT.std.pair('const string, RooDataHist*')(channel.name, obs))

        workspace.add(rooSimul)
        rooObservable = ROOT.RooArgList(channel.observable.renderRoofit(workspace))
        # that's right I don't need no CombDataSetFactory
        rooData = ROOT.RooDataHist(self.name + '_observation', 'Combined observation', rooObservable, channelCat, obsmap)
        workspace.add(rooData)
    elif rooSimul == None or rooData == None:  # noqa: E711
        raise RuntimeError('Model %r has a pdf or dataset already embedded in workspace %r' % (self, workspace))
    rooSimul = workspace.pdf(pdfName)
    rooData = workspace.data(dataName)
    return rooSimul, rooData

In [115]:
def renderCombine(self, h):
    import ROOT
    if not os.path.exists(outputPath):
        os.makedirs(outputPath)
    workspace = ROOT.RooWorkspace(self.name)
    self.renderRoofit(workspace)
    if workspace.writeToFile(os.path.join(outputPath, "%s.root" % self.name)):
        print(hi)
    for channel in self:
        channel.renderCard(os.path.join(outputPath, "%s.txt" % channel.name), self.name)
    with open(os.path.join(outputPath, "build.sh"), "w") as fout:
        cstr = " ".join("{0}={0}.txt".format(channel.name) for channel in self)
        fout.write("combineCards.py %s > %s_combined.txt\n" % (cstr, "model"))
        if self.t2w_config is None:
            fout.write("text2workspace.py model_combined.txt")
        else:
            fout.write("text2workspace.py {} model_combined.txt".format(self.t2w_config))

In [43]:
background['ttbare'].integrate('process', 'TT').integrate("systematic", 'btagUp').sum("npvs").values()

{(): array([9694.79798364, 6865.83507224, 3182.28106042, 2496.56073451,
        2451.00099807, 1729.79274506, 1189.27385461, 1136.97939632,
         796.65731001,  397.85451646,  392.42881793])}

In [44]:
background['ttbare'].integrate('process', 'TT').integrate("systematic", 'btagDown').sum("npvs").values()

{(): array([9694.79798364, 6865.83507224, 3182.28106042, 2496.56073451,
        2451.00099807, 1729.79274506, 1189.27385461, 1136.97939632,
         796.65731001,  397.85451646,  392.42881793])}

In [ ]:
workspace = ROOT.RooWorkspace(model.name)

In [ ]:
pdfName = model.name + '_simPdf'
rooSimul = workspace.pdf(pdfName)

In [ ]:
rooSimul

In [ ]:
model.renderRoofit('workspace')

In [ ]:
workspace.writeToFile()

In [ ]:
data_hists['template'].integrate('process').integrate('region', 'wjete').integrate('systematic').sum('npvs').sum('mT_corrected').values()

In [15]:
template(background, 'QCD','btagUp','wjete')[0]

array([0.99967549, 0.99988962, 1.        , 0.99960101, 0.99999349,
       0.99998535, 1.        , 1.        , 1.        , 1.        ,
       0.99984307])

In [39]:
effect_down=template(background, 'QCD','pileupDown','ttbarm')[0] 

In [40]:
nominal=template(background, 'QCD','nominal','ttbarm')[0]

In [41]:
sum(abs(effect_down-1)*nominal/np.sum(nominal))

0.37611322115700985

In [42]:
np.sum(abs(effect_down - 1) * nominal) / np.sum(nominal)

0.37611322115701

In [10]:
year='2018'
model_id = 'ControlRegionModel'
model = rl.Model(model_id)

ch_name = "wjetm" 
# Create a channel/Control region
wjetm = rl.Channel(ch_name)
# add the Channel/CR to the model
model.addChannel(wjetm)

###
# Add data distribution to the channel
###

if year == "2018":
    wjetm.setObservation(template(data, "SingleMuon", "data", "wjetm"))
else:
    wjetm.setObservation(template(data, "SingleMuon", "data","wjetm"))

###
# W(->lnu)+jets data-driven model
###

wjetm_wjetsTemplate = template(background, "W_jets", "nominal", "wjetm", read_sumw2=True)
wjetm_wjetsMC = rl.TemplateSample(
    ch_name + "_wjetsMC",
    rl.Sample.BACKGROUND,
    wjetm_wjetsTemplate
)
wjetm_wjetsMC.setParamEffect(lumi, 1.027)
# wjetm_wjetsMC.setParamEffect(trig_e, 1.01)
# wjetm_wjetsMC.setParamEffect(veto_tau, 1.03)
# wjetm_wjetsMC.setParamEffect(wjets_norm, 1.4)
# wjetm_wjetsMC.setParamEffect(jec, 1.05)
# wjetm_wjetsMC.setParamEffect(id_e, 1.02)
# wjetm_wjetsMC.setParamEffect(reco_e, 1.02)
# wjetm_wjetsMC.autoMCStats()
addBtagSyst(background, "W_jets", "wjetm", wjetm_wjetsMC)

# addVJetsSyst(background, "W_jets", "wjetm", wjetm_wjetsMC)
pileupUp=template(background, 'W_jets','pileupUp','wjetm')[0] 
pileupDown=template(background, 'W_jets','pileupDown','wjetm')[0] 
wjetm_wjetsMC.setParamEffect(pileup, pileupUp, pileupDown)
print("W_jets PU up", pileupUp)
#print("nom")
print("down", pileupDown)
wjetm.addSample(wjetm_wjetsMC)

# ###
# # top-antitop model
# ###

wjetm_ttTemplate = template(background, "TT", "nominal","wjetm", read_sumw2=True)
wjetm_ttMC = rl.TemplateSample(
    ch_name + "_ttMC",
    rl.Sample.BACKGROUND,
    wjetm_ttTemplate
)
wjetm_ttMC.setParamEffect(lumi, 1.027)
# wjetm_ttMC.setParamEffect(trig_e, 1.01)
# wjetm_ttMC.setParamEffect(veto_tau, 1.03)
# wjetm_ttMC.setParamEffect(jec, 1.05)
# wjetm_ttMC.setParamEffect(id_e, 1.02)
# wjetm_ttMC.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "TT", "wjetm", wjetm_ttMC)
# wjetm_ttMC.setParamEffect(ttMC_norm, 1.2)
wjetm.addSample(wjetm_ttMC)

# ###
# # Other MC-driven processes
# ###
# ST Background
wjetm_stTemplate = template(background, "ST", "nominal", "wjetm")
wjetm_st = rl.TemplateSample(
    ch_name + "_stMC", rl.Sample.BACKGROUND, wjetm_stTemplate
)
wjetm_st.setParamEffect(lumi, 1.027)
# wjetm_st.setParamEffect()
# wjetm_st.setParamEffect(trig_e, 1.01)
# wjetm_st.setParamEffect(veto_tau, 1.03)
# # wjetm_st.setParamEffect(st_norm, 1.2) something related to signal
# wjetm_st.setParamEffect(jec, 1.05)
# wjetm_st.setParamEffect(id_e, 1.02)
# wjetm_st.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "ST", "wjetm", wjetm_st)
trigmUp=template(background, 'ST', 'trig_mUp', 'wjetm')[0] 
trigmDown=template(background, 'ST', 'trig_mDown', 'wjetm')[0]

wjetm_st.setParamEffect(trig_m, trigmUp, trigmDown)
wjetm.addSample(wjetm_st)

# DY Jet Background
wjetm_dyjetsTemplate = template(background, "DY_jets", "nominal","wjetm")
wjetm_dyjets = rl.TemplateSample(
    ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, wjetm_dyjetsTemplate
)
wjetm_dyjets.setParamEffect(lumi, 1.027)
# wjetm_dyjets.setParamEffect(trig_e, 1.01)
# wjetm_dyjets.setParamEffect(veto_tau, 1.03)
# # wjetm_dyjets.setParamEffect(zjetsMC_norm, 1.4)
# wjetm_dyjets.setParamEffect(jec, 1.05)
# wjetm_dyjets.setParamEffect(id_e, 1.02)
# wjetm_dyjets.setParamEffect(reco_e, 1.02)
pileup_Up=template(background, 'DY_jets','pileupUp','wjetm')[0] 
pileup_Down=template(background, "DY_jets",'pileupDown','wjetm')[0] 


#print("DY PU up", pileup_Up)
# #print("nom")
#print("down", pileup_Down)
wjetm_dyjets.setParamEffect(pileup, pileup_Up, pileup_Down)
addBtagSyst(background, "DY_jets", "wjetm", wjetm_dyjets)
addVJetsSyst(background,"DY_jets", "wjetm", wjetm_dyjets)
#print("hi")
#print(wjetm_dyjets.getParamEffect(ew1))
wjetm.addSample(wjetm_dyjets)


# V Jets Background
wjetm_vvTemplate = template(background, "VV", "nominal", "wjetm")
wjetm_vv = rl.TemplateSample(
    ch_name + "_vvMC", rl.Sample.BACKGROUND, wjetm_vvTemplate
)
wjetm_vv.setParamEffect(lumi, 1.027)
# wjetm_vv.setParamEffect(trig_e, 1.01)
# wjetm_vv.setParamEffect(veto_tau, 1.03)
# # wjetm_vv.setParamEffect(vv_norm, 1.2)
# wjetm_vv.setParamEffect(jec, 1.05)
# wjetm_vv.setParamEffect(id_e, 1.02)
# wjetm_vv.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "VV", "wjetm", wjetm_vv)
wjetm.addSample(wjetm_vv)

# QCD background
wjetm_qcdTemplate = template(background, "QCD", "nominal", "wjetm")
wjetm_qcd = rl.TemplateSample(
    ch_name + "_qcdMC", rl.Sample.BACKGROUND, wjetm_qcdTemplate
)
wjetm_qcd.setParamEffect(lumi, 1.027)
pileup_Up=template(background, 'QCD','pileupUp','wjetm')[0] 
pileup_Down=template(background, "QCD",'pileupDown','wjetm')[0] 


#print("qcd PU up", pileup_Up)
# #print("nom")
#print("down", pileup_Down)
wjetm_qcd.setParamEffect(pileup, pileup_Up, pileup_Down)
# wjetm_qcd.setParamEffect(trig_e, 1.01)
# wjetm_qcd.setParamEffect(veto_tau, 1.03)
# # wjetm_qcd.setParamEffect(qcde_norm, 2.0)
# wjetm_qcd.setParamEffect(jec, 1.05)
# wjetm_qcd.setParamEffect(id_e, 1.02)
# wjetm_qcd.setParamEffect(reco_e, 1.02)
addBtagSyst(background, "QCD", "wjetm", wjetm_qcd)
wjetm.addSample(wjetm_qcd)


for s in signal['wjetm'].identifiers('process'):
#     #print(s)
    if 'Mphi1995' not in str(s): continue
    wjetm_signalTemplate = template(signal,s,'nominal','wjetm')
    wjetm_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, wjetm_signalTemplate)
    wjetm_signal.setParamEffect(lumi, 1.027)
#     sr_signal.setParamEffect(trig_met, 1.01)
#     sr_signal.setParamEffect(veto_tau, 1.03)
#     sr_signal.setParamEffect(jec, 1.05)
    btagUp=template(signal, 'Mphi1995_Mchi1000','btagUp','wjetm')[0] 
    btagDown=template(signal, 'Mphi1995_Mchi1000','btagDown','wjetm')[0] 
#     #print("btag")
#     #print(btagDown)
    wjetm_signal.setParamEffect(btag, btagUp, btagDown)
#     #print(wjetm_signal.getParamEffect(btag))
    trigmUp=template(signal, 'Mphi1995_Mchi1000', 'trig_mUp', 'wjetm')[0] 
    trigmDown=template(signal, 'Mphi1995_Mchi1000', 'trig_mDown', 'wjetm')[0]
    wjetm_signal.setParamEffect(trig_m, trigmUp, trigmDown)
#     #print("trig")
#     #print("trigmUp",trigmUp)
#     #print(wjetm_signal.getParamEffect(trig_m))
        #id Muon
    id_mUp=template(signal, s, 'id_mUp', 'wjetm')[0] 
    id_mDown=template(signal, s, 'id_mDown', 'wjetm')[0]
    wjetm_signal.setParamEffect(id_m, id_mUp, id_mDown)
    pileup_Up=template(signal, s,'pileupUp','wjetm')[0] 
    pileup_Down=template(signal, s,'pileupDown','wjetm')[0] 
    wjetm_signal.setParamEffect(pileup, pileup_Up, pileup_Down)

    #print("qcd PU up", pileup_Up)
    # #print("nom")
    #print("down", pileup_Down)

#     #print("id")
    #print(pileup_Down)
    wjetm.addSample(wjetm_signal)
    
    
    

W_jets PU up [0.95703642 0.94959824 0.95243282 1.01151968 0.99833867 1.03487551
 1.01659317 0.97093242 0.99859661 1.01504948 0.98259553]
down [0.96643794 0.93192665 0.95337324 1.05372704 1.07256106 1.06364375
 1.15042586 1.04030616 1.09716878 1.12756608 1.06199435]


# ttbare

In [ ]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mT_corrected_binning_dict = {
    "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
}



def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z_jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi1995_Mchi1000"] = ("Mphi-1995_Mchi-1000")
    sig_map["Mphi2000_Mchi1500"] = ("Mphi-2000_Mchi-1500")
    sig_map["Mphi2000_Mchi500"] = ("Mphi-2000_Mchi-500")
#     haven't processed signal yet
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mchi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists, "sig":signal_hists}

    return hists


def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
        histogram = dictionary[region].integrate("process", process)
#         print(histogram.integrate("systematic", "nominal").sum('npvs').values()[()])
        nominal=histogram.integrate("systematic", "nominal").sum('npvs').values()[()]
        up=histogram.integrate("systematic", string+"Up").sum('npvs').values()[()]
        down=histogram.integrate("systematic",string+"Down").sum('npvs').values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
    addSyst(dictionary, process, region, templ, ew2W, "ew2W")
    addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
    addSyst(dictionary, process, region, templ, ew3W, "ew3W")
    addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
    addSyst(dictionary, process, region, templ, mix, "mix")
    addSyst(dictionary, process, region, templ, qcd1, "qcd1")
    addSyst(dictionary, process, region, templ, qcd2, "qcd2")
    addSyst(dictionary, process, region, templ, qcd3, "qcd3")
    addSyst(dictionary, process, region, templ, muR, "muR")
    addSyst(dictionary, process, region, templ, muF, "muF")
    
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process).sum('npvs')
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT_corrected")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT_corrected", sumw2)
    return (output, binning, "mT_corrected")

def model(year):

#     model_id = year + category + "recoil" + str(recoil)
#     print(model_id)
#     model = rl.Model(str(s) + model_id)



    model_id = 'w'
    model = rl.Model(model_id)
    ch_name = "ttbare" 
    # Create a channel/Control region
    ttbare = rl.Channel(ch_name)
    # add the Channel/CR to the model
    model.addChannel(ttbare)
    
    ###
    # Add data distribution to the channel
    ###

    if year == "2018":
        ttbare.setObservation(template(data, "EGamma", "data", "ttbare"))
    else:
        ttbare.setObservation(template(data, "SingleElectron", "data","ttbare"))
        
    #template for signal
    for s in signal['ttbare'].identifiers('process'):
        print(s)
        if 'Mphi1995' not in str(s): continue
        ttbare_signalTemplate = template(signal,s,'nominal','ttbare')
        ttbare_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, ttbare_signalTemplate)
        ttbare_signal.setParamEffect(lumi, 1.027)
    #     sr_signal.setParamEffect(trig_met, 1.01)
    #     sr_signal.setParamEffect(veto_tau, 1.03)
    #     sr_signal.setParamEffect(jec, 1.05)
        #btag
        btagUp=template(signal, s,'btagUp','ttbare')[0]
        btagDown=template(signal, s,'btagDown','ttbare')[0]
        ttbare_signal.setParamEffect(btag, btagUp, btagDown)
        #trigger Electron
        trigeUp=template(signal, s, 'trig_eUp', 'ttbare')[0] 
        trigeDown=template(signal, s, 'trig_eDown', 'ttbare')[0]
        ttbare_signal.setParamEffect(trig_e, trigeUp, trigeDown)
        #id Electron
        id_eUp=template(signal, s, 'id_eUp', 'ttbare')[0] 
        id_eDown=template(signal, s, 'id_eDown', 'ttbare')[0]
        ttbare_signal.setParamEffect(id_e, id_eUp, id_eDown)
        #pileup
        pileupUp=template(signal, s,'pileupUp','ttbare')[0] 
        pileupDown=template(signal, s,'pileupDown','ttbare')[0] 
        ttbare_signal.setParamEffect(pileup, pileupUp, pileupDown)
        
        #reco Electron
        recoeUp=template(signal, s, 'reco_eUp', 'ttbare')[0] 
        recoDown=template(signal, s, 'reco_eDown', 'ttbare')[0]
        ttbare_signal.setParamEffect(trig_e, recoeUp, recoDown)
        
        ttbare.addSample(ttbare_signal)
    ###
    # W(->lnu)+jets data-driven model
    ###

    ttbare_wjetsTemplate = template(background, "W+jets", "nominal", "ttbare", read_sumw2=True)
    ttbare_wjetsMC = rl.TemplateSample(
        ch_name + "_wjetsMC",
        rl.Sample.BACKGROUND,
        ttbare_wjetsTemplate
    )
    ttbare_wjetsMC.setParamEffect(lumi, 1.027)
#     ttbare_wjetsMC.setParamEffect(trig_e, 1.01)
#     ttbare_wjetsMC.setParamEffect(veto_tau, 1.03)
#     ttbare_wjetsMC.setParamEffect(wjets_norm, 1.4)
#     ttbare_wjetsMC.setParamEffect(jec, 1.05)
#     ttbare_wjetsMC.setParamEffect(id_e, 1.02)
#     ttbare_wjetsMC.setParamEffect(reco_e, 1.02)
    #ttbare_wjetsMC.autoMCStats()
    addBtagSyst(background, "W+jets", "ttbare", ttbare_wjetsMC)
    #trigger Electron
    trigeUp=template(background, 'W+jets', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'W+jets', 'trig_eDown', 'ttbare')[0]
    ttbare_wjetsMC.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "W+jets", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "W+jets", 'id_eDown', 'ttbare')[0]
    ttbare_wjetsMC.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'W+jets','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'W+jets','pileupDown','ttbare')[0] 
    ttbare_wjetsMC.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'W+jets', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(background, 'W+jets', 'reco_eDown', 'ttbare')[0]
    ttbare_wjetsMC.setParamEffect(trig_e, recoeUp, recoDown)
    # addVJetsSyst(background, "W+jets", "ttbare", ttbare_wjetsMC)
    ttbare.addSample(ttbare_wjetsMC)

    # ###
    # # top-antitop model
    # ###

    ttbare_ttTemplate = template(background, "TT", "nominal","ttbare", read_sumw2=True)
    ttbare_ttMC = rl.TemplateSample(
        ch_name + "_ttMC",
        rl.Sample.BACKGROUND,
        ttbare_ttTemplate
    )
    ttbare_ttMC.setParamEffect(lumi, 1.027)
#     ttbare_ttMC.setParamEffect(trig_e, 1.01)
#     ttbare_ttMC.setParamEffect(veto_tau, 1.03)
#     ttbare_ttMC.setParamEffect(jec, 1.05)
#     ttbare_ttMC.setParamEffect(id_e, 1.02)
#     ttbare_ttMC.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "TT", "ttbare", ttbare_ttMC)
    #trigger Electron
    trigeUp=template(background, 'TT', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'TT', 'trig_eDown', 'ttbare')[0]
    ttbare_ttMC.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "TT", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "TT", 'id_eDown', 'ttbare')[0]
    ttbare_ttMC.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'TT','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'TT','pileupDown','ttbare')[0] 
    ttbare_ttMC.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'TT', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(background, 'TT', 'reco_eDown', 'ttbare')[0]
    ttbare_ttMC.setParamEffect(trig_e, recoeUp, recoDown)
    # ttbare_ttMC.setParamEffect(ttMC_norm, 1.2)
    ttbare.addSample(ttbare_ttMC)

    # ###
    # # Other MC-driven processes
    # ###
    # ST Background
    ttbare_stTemplate = template(background, "ST", "nominal", "ttbare")
    ttbare_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, ttbare_stTemplate
    )
    ttbare_st.setParamEffect(lumi, 1.027)
#     ttbare_st.setParamEffect(trig_e, 1.01)
#     ttbare_st.setParamEffect(veto_tau, 1.03)
#     ttbare_st.setParamEffect(st_norm, 1.2) something related to signal
#     ttbare_st.setParamEffect(jec, 1.05)
#     ttbare_st.setParamEffect(id_e, 1.02)
#     ttbare_st.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "ST", "ttbare", ttbare_st)
    #trigger Electron
    trigeUp=template(background, 'ST', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'ST', 'trig_eDown', 'ttbare')[0]
    ttbare_st.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "ST", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "ST", 'id_eDown', 'ttbare')[0]
    ttbare_st.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'ST','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'ST','pileupDown','ttbare')[0] 
    ttbare_st.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'ST', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(background, 'ST', 'reco_eDown', 'ttbare')[0]
    ttbare_st.setParamEffect(trig_e, recoeUp, recoDown)
    ttbare.addSample(ttbare_st)

    # DY Jet Background
    ttbare_dyjetsTemplate = template(background, "DY+jets", "nominal","ttbare")
    ttbare_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbare_dyjetsTemplate
    )
    ttbare_dyjets.setParamEffect(lumi, 1.027)
#     ttbare_dyjets.setParamEffect(trig_e, 1.01)
#     ttbare_dyjets.setParamEffect(veto_tau, 1.03)
#     ttbare_dyjets.setParamEffect(zjetsMC_norm, 1.4)
#     ttbare_dyjets.setParamEffect(jec, 1.05)
#     ttbare_dyjets.setParamEffect(id_e, 1.02)
#     ttbare_dyjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "DY+jets", "ttbare", ttbare_dyjets)
    addVJetsSyst(background,"DY+jets", "ttbare", ttbare_dyjets)
    #trigger Electron
    trigeUp=template(background, 'DY+jets', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'DY+jets', 'trig_eDown', 'ttbare')[0]
    ttbare_dyjets.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "DY+jets", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "DY+jets", 'id_eDown', 'ttbare')[0]
    ttbare_dyjets.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'DY+jets','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'DY+jets','pileupDown','ttbare')[0] 
    ttbare_dyjets.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'DY+jets', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(background, 'DY+jets', 'reco_eDown', 'ttbare')[0]
    ttbare_dyjets.setParamEffect(trig_e, recoeUp, recoDown)
    ttbare.addSample(ttbare_dyjets)


    # V Jets Background
    ttbare_vvTemplate = template(background, "VV", "nominal", "ttbare")
    ttbare_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbare_vvTemplate
    )
    ttbare_vv.setParamEffect(lumi, 1.027)
#     ttbare_vv.setParamEffect(trig_e, 1.01)
#     ttbare_vv.setParamEffect(veto_tau, 1.03)
#     ttbare_vv.setParamEffect(vv_norm, 1.2)
#     ttbare_vv.setParamEffect(jec, 1.05)
#     ttbare_vv.setParamEffect(id_e, 1.02)
#     ttbare_vv.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "VV", "ttbare", ttbare_vv)
    #trigger Electron
    trigeUp=template(background, 'VV', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'VV', 'trig_eDown', 'ttbare')[0]
    ttbare_vv.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "VV", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "VV", 'id_eDown', 'ttbare')[0]
    ttbare_vv.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'VV','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'VV','pileupDown','ttbare')[0] 
    ttbare_vv.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'VV', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(background, 'VV', 'reco_eDown', 'ttbare')[0]
    ttbare_vv.setParamEffect(trig_e, recoeUp, recoDown)
    
    ttbare.addSample(ttbare_vv)

    # QCD background
    ttbare_qcdTemplate = template(background, "QCD", "nominal", "ttbare")
    ttbare_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbare_qcdTemplate
    )
    ttbare_qcd.setParamEffect(lumi, 1.027)
#     ttbare_qcd.setParamEffect(trig_e, 1.01)
#     ttbare_qcd.setParamEffect(veto_tau, 1.03)
#     ttbare_qcd.setParamEffect(qcde_norm, 2.0)
#     ttbare_qcd.setParamEffect(jec, 1.05)
#     ttbare_qcd.setParamEffect(id_e, 1.02)
#     ttbare_qcd.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "QCD", "ttbare", ttbare_qcd)
    #trigger Electron
    trigeUp=template(background, 'QCD', 'trig_eUp', 'ttbare')[0] 
    trigeDown=template(background, 'QCD', 'trig_eDown', 'ttbare')[0]
    ttbare_qcd.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "QCD", 'id_eUp', 'ttbare')[0] 
    id_eDown=template(background, "QCD", 'id_eDown', 'ttbare')[0]
    ttbare_qcd.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'QCD','pileupUp','ttbare')[0] 
    pileupDown=template(background, 'QCD','pileupDown','ttbare')[0] 
    ttbare_qcd.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'QCD', 'reco_eUp', 'ttbare')[0] 
    recoDown=template(background, 'QCD', 'reco_eDown', 'ttbare')[0]
    ttbare_qcd.setParamEffect(trig_e, recoeUp, recoDown)
    
    ttbare.addSample(ttbare_qcd)
    return model

if __name__ == "__main__":
    if not os.path.exists("datacards"):
        os.mkdir("datacards")
    parser = OptionParser()
    parser.add_option("-y", "--year", help="year", dest="year", default="")
    (options, args) = parser.parse_args()
    year = options.year
    year = '2018'

    mT_corrected_binning_dict = {
        "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
    }
    mT_corrected_binning = mT_corrected_binning_dict[year]
    mT_correctedbins = np.array(mT_corrected_binning)
    nmT_corrected = len(mT_correctedbins) - 1

    ###
    # Extract histograms from input file
    ###

    print("Grouping histograms")
    hists = load("hists/UL_lep_monotop_v1_test_" + year + ".scaled")
    hists = remap_histograms(hists)
    data_hists = hists["data"]
    bkg_hists = hists["bkg"]
    signal_hists = hists["sig"]

    ###
    # Preparing histograms for fit
    ##

    data = {}
    for r in data_hists["template"].identifiers("region"):
        data[str(r)] = data_hists["template"].integrate("region", r)

    background = {}
    for r in bkg_hists["template"].identifiers("region"):
        background[str(r)] = bkg_hists["template"].integrate("region", r)

    signal = {}
    for r in signal_hists["template"].identifiers("region"):
        signal[str(r)] = signal_hists["template"].integrate("region", r)
    ###
    ###
    # Setting up systematics
    ###
    ###
    lumi = rl.NuisanceParameter("lumi" + year, "lnN")
    zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
    wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
    tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
    id_e = rl.NuisanceParameter("id_e" + year, "shape")
    id_mu = rl.NuisanceParameter("id_mu" + year, "lnN")
    id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
    reco_e = rl.NuisanceParameter("reco_e" + year, "shape")
    iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
    trig_e = rl.NuisanceParameter("trig_e" + year, "shape")
    trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
    trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
    veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
    jec = rl.NuisanceParameter("jec" + year, "lnN")
    #btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
    btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
    ew1 = rl.NuisanceParameter("ew1", "lnN")
    #ew2G = rl.NuisanceParameter("ew2G", "lnN")
    ew2W = rl.NuisanceParameter("ew2W", "lnN")
    ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
    #ew3G = rl.NuisanceParameter("ew3G", "lnN")
    ew3W = rl.NuisanceParameter("ew3W", "lnN")
    ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
    mix = rl.NuisanceParameter("mix", "lnN")
    #muF = rl.NuisanceParameter("muF", "lnN")
    #muR = rl.NuisanceParameter("muR", "lnN")
    qcd1 = rl.NuisanceParameter("qcd1", "lnN")
    qcd2 = rl.NuisanceParameter("qcd2", "lnN")
    qcd3 = rl.NuisanceParameter("qcd3", "lnN")
    whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
    zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
    #ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")
    pileup = rl.NuisanceParameter("pileup"+year, "shape")
    
    with open(
        "data/"+"test"+year+".model",
        "wb",) as fout:
        pickle.dump(model(year), fout, protocol=2)

# ttbarm

In [ ]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mT_corrected_binning_dict = {
    "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
}



def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z_jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi1995_Mchi1000"] = ("Mphi-1995_Mchi-1000")
    sig_map["Mphi2000_Mchi1500"] = ("Mphi-2000_Mchi-1500")
    sig_map["Mphi2000_Mchi500"] = ("Mphi-2000_Mchi-500")
#     haven't processed signal yet
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mchi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists, "sig":signal_hists}

    return hists


def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
        histogram = dictionary[region].integrate("process", process)
#         print(histogram.integrate("systematic", "nominal").sum('npvs').values()[()])
        nominal=histogram.integrate("systematic", "nominal").sum('npvs').values()[()]
        up=histogram.integrate("systematic", string+"Up").sum('npvs').values()[()]
        down=histogram.integrate("systematic",string+"Down").sum('npvs').values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
    addSyst(dictionary, process, region, templ, ew2W, "ew2W")
    addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
    addSyst(dictionary, process, region, templ, ew3W, "ew3W")
    addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
    addSyst(dictionary, process, region, templ, mix, "mix")
    addSyst(dictionary, process, region, templ, qcd1, "qcd1")
    addSyst(dictionary, process, region, templ, qcd2, "qcd2")
    addSyst(dictionary, process, region, templ, qcd3, "qcd3")
    addSyst(dictionary, process, region, templ, muR, "muR")
    addSyst(dictionary, process, region, templ, muF, "muF")
    
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process).sum('npvs')
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT_corrected")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT_corrected", sumw2)
    return (output, binning, "mT_corrected")

def model(year):

#     model_id = year + category + "recoil" + str(recoil)
#     print(model_id)
#     model = rl.Model(str(s) + model_id)



    model_id = 'w'
    model = rl.Model(model_id)
    ch_name = "ttbarm" 
    # Create a channel/Control region
    ttbarm = rl.Channel(ch_name)
    # add the Channel/CR to the model
    model.addChannel(ttbarm)
    
    ###
    # Add data distribution to the channel
    ###

    if year == "2018":
        ttbarm.setObservation(template(data, "EGamma", "data", "ttbarm"))
    else:
        ttbarm.setObservation(template(data, "SingleElectron", "data","ttbarm"))
        
    #template for signal
    for s in signal['ttbarm'].identifiers('process'):
        print(s)
        if 'Mphi1995' not in str(s): continue
        ttbarm_signalTemplate = template(signal,s,'nominal','ttbarm')
        ttbarm_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, ttbarm_signalTemplate)
        ttbarm_signal.setParamEffect(lumi, 1.027)
    #     sr_signal.setParamEffect(trig_met, 1.01)
    #     sr_signal.setParamEffect(veto_tau, 1.03)
    #     sr_signal.setParamEffect(jec, 1.05)
        #btag
        btagUp=template(signal, s,'btagUp','ttbarm')[0]
        btagDown=template(signal, s,'btagDown','ttbarm')[0]
        ttbarm_signal.setParamEffect(btag, btagUp, btagDown)
        #trigger Muon
        trigm_uUp=template(signal, 'W+jets', 'trig_mUp', 'ttbarm')[0] 
        trig_muDown=template(signal, 'W+jets', 'trig_mDown', 'ttbarm')[0]
        ttbarm_signal.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
        #id Muon
        id_mUp=template(signal, "W+jets", 'id_mUp', 'ttbarm')[0] 
        id_mDown=template(signal, "W+jets", 'id_mDown', 'ttbarm')[0]
        ttbarm_signal.setParamEffect(id_m, id_mUp, id_mDown)
        #pileup
        pileup_Up=template(signal, 'W+jets','pileupUp','ttbarm')[0] 
        pileup_Down=template(signal, 'W+jets','pileupDown','ttbarm')[0] 
        ttbarm_signal.setParamEffect(pileup, pileup_Up, pileup_Down)
        #iso Muon
        iso_mUp=template(signal, 'W+jets', 'isolation_mUp', 'ttbarm')[0] 
        iso_mDown=template(signal, 'W+jets', 'isolation_mDown', 'ttbarm')[0]
        ttbarm_signal.setParamEffect(isolation_m, iso_mUp, iso_mDown)
        ttbarm.addSample(ttbare_signal)
    ###
    # W(->lnu)+jets data-driven model
    ###

    ttbarm_wjetsTemplate = template(background, "W+jets", "nominal", "ttbarm", read_sumw2=True)
    ttbarm_wjetsMC = rl.TemplateSample(
        ch_name + "_wjetsMC",
        rl.Sample.BACKGROUND,
        ttbarm_wjetsTemplate
    )
    ttbarm_wjetsMC.setParamEffect(lumi, 1.027)
#     ttbarm_wjetsMC.setParamEffect(trig_e, 1.01)
#     ttbarm_wjetsMC.setParamEffect(veto_tau, 1.03)
#     ttbarm_wjetsMC.setParamEffect(wjets_norm, 1.4)
#     ttbarm_wjetsMC.setParamEffect(jec, 1.05)
#     ttbarm_wjetsMC.setParamEffect(id_e, 1.02)
#     ttbarm_wjetsMC.setParamEffect(reco_e, 1.02)
    #ttbarm_wjetsMC.autoMCStats()
    addBtagSyst(background, "W+jets", "ttbarm", ttbarm_wjetsMC)
    #trigger Muon
    trigm_uUp=template(background, 'W+jets', 'trig_mUp', 'ttbarm')[0] 
    trig_muDown=template(background, 'W+jets', 'trig_mDown', 'ttbarm')[0]
    ttbarm_wjetsMC.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
    #id Muon
    id_mUp=template(background, "W+jets", 'id_mUp', 'ttbarm')[0] 
    id_mDown=template(background, "W+jets", 'id_mDown', 'ttbarm')[0]
    ttbarm_wjetsMC.setParamEffect(id_m, id_mUp, id_mDown)
    #pileup
    pileup_Up=template(background, 'W+jets','pileupUp','ttbarm')[0] 
    pileup_Down=template(background, 'W+jets','pileupDown','ttbarm')[0] 
    ttbarm_wjetsMC.setParamEffect(pileup, pileup_Up, pileup_Down)
    #iso Muon
    iso_mUp=template(background, 'W+jets', 'isolation_mUp', 'ttbarm')[0] 
    iso_mDown=template(background, 'W+jets', 'isolation_mDown', 'ttbarm')[0]
    ttbarm_wjetsMC.setParamEffect(isolation_m, iso_mUp, iso_mDown)
    # addVJetsSyst(background, "W+jets", "ttbarm", ttbarm_wjetsMC)
    ttbarm.addSample(ttbarm_wjetsMC)

    # ###
    # # top-antitop model
    # ###

    ttbarm_ttTemplate = template(background, "TT", "nominal","ttbarm", read_sumw2=True)
    ttbarm_ttMC = rl.TemplateSample(
        ch_name + "_ttMC",
        rl.Sample.BACKGROUND,
        ttbarm_ttTemplate
    )
    ttbarm_ttMC.setParamEffect(lumi, 1.027)
#     ttbarm_ttMC.setParamEffect(trig_e, 1.01)
#     ttbarm_ttMC.setParamEffect(veto_tau, 1.03)
#     ttbarm_ttMC.setParamEffect(jec, 1.05)
#     ttbarm_ttMC.setParamEffect(id_e, 1.02)
#     ttbarm_ttMC.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "TT", "ttbarm", ttbarm_ttMC)
    #trigger Muon
    trigm_uUp=template(background, 'TT', 'trig_mUp', 'ttbarm')[0] 
    trig_muDown=template(background, 'TT', 'trig_mDown', 'ttbarm')[0]
    ttbarm_ttMC.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
    #id Muon
    id_mUp=template(background, "TT", 'id_mUp', 'ttbarm')[0] 
    id_mDown=template(background, "TT", 'id_mDown', 'ttbarm')[0]
    ttbarm_ttMC.setParamEffect(id_m, id_mUp, id_mDown)
    #pileup
    pileup_Up=template(background, 'TT','pileupUp','ttbarm')[0] 
    pileup_Down=template(background, 'TT','pileupDown','ttbarm')[0] 
    ttbarm_ttMC.setParamEffect(pileup, pileup_Up, pileup_Down)
    #iso Muon
    iso_mUp=template(background, 'TT', 'isolation_mUp', 'ttbarm')[0] 
    iso_mDown=template(background, 'TT', 'isolation_mDown', 'ttbarm')[0]
    ttbarm_ttMC.setParamEffect(isolation_m, iso_mUp, iso_mDown)
    # ttbarm_ttMC.setParamEffect(ttMC_norm, 1.2)
    ttbarm.addSample(ttbarm_ttMC)

    # ###
    # # Other MC-driven processes
    # ###
    # ST Background
    ttbarm_stTemplate = template(background, "ST", "nominal", "ttbarm")
    ttbarm_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, ttbarm_stTemplate
    )
    ttbarm_st.setParamEffect(lumi, 1.027)
#     ttbarm_st.setParamEffect(trig_e, 1.01)
#     ttbarm_st.setParamEffect(veto_tau, 1.03)
#     ttbarm_st.setParamEffect(st_norm, 1.2) something related to signal
#     ttbarm_st.setParamEffect(jec, 1.05)
#     ttbarm_st.setParamEffect(id_e, 1.02)
#     ttbarm_st.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "ST", "ttbarm", ttbarm_st)
    #trigger Muon
    trigm_uUp=template(background, 'ST', 'trig_mUp', 'ttbarm')[0] 
    trig_muDown=template(background, 'ST', 'trig_mDown', 'ttbarm')[0]
    ttbarm_st.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
    #id Muon
    id_mUp=template(background, "ST", 'id_mUp', 'ttbarm')[0] 
    id_mDown=template(background, "ST", 'id_mDown', 'ttbarm')[0]
    ttbarm_st.setParamEffect(id_m, id_mUp, id_mDown)
    #pileup
    pileup_Up=template(background, 'ST','pileupUp','ttbarm')[0] 
    pileup_Down=template(background, 'ST','pileupDown','ttbarm')[0] 
    ttbarm_st.setParamEffect(pileup, pileup_Up, pileup_Down)
    #iso Muon
    iso_mUp=template(background, 'ST', 'isolation_mUp', 'ttbarm')[0] 
    iso_mDown=template(background, 'ST', 'isolation_mDown', 'ttbarm')[0]
    ttbarm_st.setParamEffect(isolation_m, iso_mUp, iso_mDown)
    ttbarm.addSample(ttbarm_st)

    # DY Jet Background
    ttbarm_dyjetsTemplate = template(background, "DY+jets", "nominal","ttbarm")
    ttbarm_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbarm_dyjetsTemplate
    )
    ttbarm_dyjets.setParamEffect(lumi, 1.027)
#     ttbarm_dyjets.setParamEffect(trig_e, 1.01)
#     ttbarm_dyjets.setParamEffect(veto_tau, 1.03)
#     ttbarm_dyjets.setParamEffect(zjetsMC_norm, 1.4)
#     ttbarm_dyjets.setParamEffect(jec, 1.05)
#     ttbarm_dyjets.setParamEffect(id_e, 1.02)
#     ttbarm_dyjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "DY+jets", "ttbarm", ttbarm_dyjets)
    addVJetsSyst(background,"DY+jets", "ttbarm", ttbarm_dyjets)
    #trigger Muon
    trigm_uUp=template(background, 'DY+jets', 'trig_mUp', 'ttbarm')[0] 
    trig_muDown=template(background, 'DY+jets', 'trig_mDown', 'ttbarm')[0]
    ttbarm_dyjets.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
    #id Muon
    id_mUp=template(background, "DY+jets", 'id_mUp', 'ttbarm')[0] 
    id_mDown=template(background, "DY+jets", 'id_mDown', 'ttbarm')[0]
    ttbarm_dyjets.setParamEffect(id_m, id_mUp, id_mDown)
    #pileup
    pileup_Up=template(background, 'DY+jets','pileupUp','ttbarm')[0] 
    pileup_Down=template(background, 'DY+jets','pileupDown','ttbarm')[0] 
    ttbarm_dyjets.setParamEffect(pileup, pileup_Up, pileup_Down)
    #iso Muon
    iso_mUp=template(background, 'DY+jets', 'isolation_mUp', 'ttbarm')[0] 
    iso_mDown=template(background, 'DY+jets', 'isolation_mDown', 'ttbarm')[0]
    ttbarm_dyjets.setParamEffect(isolation_m, iso_mUp, iso_mDown)
    ttbarm.addSample(ttbarm_dyjets)


    # V Jets Background
    ttbarm_vvTemplate = template(background, "VV", "nominal", "ttbarm")
    ttbarm_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbarm_vvTemplate
    )
    ttbarm_vv.setParamEffect(lumi, 1.027)
#     ttbarm_vv.setParamEffect(trig_e, 1.01)
#     ttbarm_vv.setParamEffect(veto_tau, 1.03)
#     ttbarm_vv.setParamEffect(vv_norm, 1.2)
#     ttbarm_vv.setParamEffect(jec, 1.05)
#     ttbarm_vv.setParamEffect(id_e, 1.02)
#     ttbarm_vv.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "VV", "ttbarm", ttbarm_vv)
    #trigger Muon
    trigm_uUp=template(background, 'VV', 'trig_mUp', 'ttbarm')[0] 
    trig_muDown=template(background, 'VV', 'trig_mDown', 'ttbarm')[0]
    ttbarm_vv.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
    #id Muon
    id_mUp=template(background, "VV", 'id_mUp', 'ttbarm')[0] 
    id_mDown=template(background, "VV", 'id_mDown', 'ttbarm')[0]
    ttbarm_vv.setParamEffect(id_m, id_mUp, id_mDown)
    #pileup
    pileup_Up=template(background, 'VV','pileupUp','ttbarm')[0] 
    pileup_Down=template(background, 'VV','pileupDown','ttbarm')[0] 
    ttbarm_vv.setParamEffect(pileup, pileup_Up, pileup_Down)
    #iso Muon
    iso_mUp=template(background, 'VV', 'isolation_mUp', 'ttbarm')[0] 
    iso_mDown=template(background, 'VV', 'isolation_mDown', 'ttbarm')[0]
    ttbarm_vv.setParamEffect(isolation_m, iso_mUp, iso_mDown)
    
    ttbarm.addSample(ttbarm_vv)

    # QCD background
    ttbarm_qcdTemplate = template(background, "QCD", "nominal", "ttbarm")
    ttbarm_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbarm_qcdTemplate
    )
    ttbarm_qcd.setParamEffect(lumi, 1.027)
#     ttbarm_qcd.setParamEffect(trig_e, 1.01)
#     ttbarm_qcd.setParamEffect(veto_tau, 1.03)
#     ttbarm_qcd.setParamEffect(qcde_norm, 2.0)
#     ttbarm_qcd.setParamEffect(jec, 1.05)
#     ttbarm_qcd.setParamEffect(id_e, 1.02)
#     ttbarm_qcd.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "QCD", "ttbarm", ttbarm_qcd)
    #trigger Muon
    trigm_uUp=template(background, 'QCD', 'trig_mUp', 'ttbarm')[0] 
    trig_muDown=template(background, 'QCD', 'trig_mDown', 'ttbarm')[0]
    ttbarm_qcd.setParamEffect(trig_mu, trigm_uUp, trig_muDown)
    #id Muon
    id_mUp=template(background, "QCD", 'id_mUp', 'ttbarm')[0] 
    id_mDown=template(background, "QCD", 'id_mDown', 'ttbarm')[0]
    ttbarm_qcd.setParamEffect(id_m, id_mUp, id_mDown)
    #pileup
    pileup_Up=template(background, 'QCD','pileupUp','ttbarm')[0] 
    pileup_Down=template(background, 'QCD','pileupDown','ttbarm')[0] 
    ttbarm_qcd.setParamEffect(pileup, pileup_Up, pileup_Down)
    #iso Muon
    iso_mUp=template(background, 'QCD', 'isolation_mUp', 'ttbarm')[0] 
    iso_mDown=template(background, 'QCD', 'isolation_mDown', 'ttbarm')[0]
    ttbarm_qcd.setParamEffect(isolation_m, iso_mUp, iso_mDown)
    
    ttbarm.addSample(ttbarm_qcd)
    return model

if __name__ == "__main__":
    if not os.path.exists("datacards"):
        os.mkdir("datacards")
    parser = OptionParser()
    parser.add_option("-y", "--year", help="year", dest="year", default="")
    (options, args) = parser.parse_args()
    year = options.year
    year = '2018'

    mT_corrected_binning_dict = {
        "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
    }
    mT_corrected_binning = mT_corrected_binning_dict[year]
    mT_correctedbins = np.array(mT_corrected_binning)
    nmT_corrected = len(mT_correctedbins) - 1

    ###
    # Extract histograms from input file
    ###

    print("Grouping histograms")
    hists = load("hists/UL_lep_monotop_v1_test_" + year + ".scaled")
    hists = remap_histograms(hists)
    data_hists = hists["data"]
    bkg_hists = hists["bkg"]
    signal_hists = hists["sig"]

    ###
    # Preparing histograms for fit
    ##

    data = {}
    for r in data_hists["template"].identifiers("region"):
        data[str(r)] = data_hists["template"].integrate("region", r)

    background = {}
    for r in bkg_hists["template"].identifiers("region"):
        background[str(r)] = bkg_hists["template"].integrate("region", r)

    signal = {}
    for r in signal_hists["template"].identifiers("region"):
        signal[str(r)] = signal_hists["template"].integrate("region", r)
    ###
    ###
    # Setting up systematics
    ###
    ###
    lumi = rl.NuisanceParameter("lumi" + year, "lnN")
    zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
    wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
    tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
    id_e = rl.NuisanceParameter("id_e" + year, "shape")
    id_m = rl.NuisanceParameter("id_m" + year, "lnN")
    id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
    reco_e = rl.NuisanceParameter("reco_e" + year, "shape")
    isolation_m = rl.NuisanceParameter("isolation_m" + year, "shape")
    trig_e = rl.NuisanceParameter("trig_e" + year, "shape")
    trig_m = rl.NuisanceParameter("trig_m" + year, "shape")

    veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
    jec = rl.NuisanceParameter("jec" + year, "lnN")
    #btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
    btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
    ew1 = rl.NuisanceParameter("ew1", "lnN")
    #ew2G = rl.NuisanceParameter("ew2G", "lnN")
    ew2W = rl.NuisanceParameter("ew2W", "lnN")
    ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
    #ew3G = rl.NuisanceParameter("ew3G", "lnN")
    ew3W = rl.NuisanceParameter("ew3W", "lnN")
    ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
    mix = rl.NuisanceParameter("mix", "lnN")
    #muF = rl.NuisanceParameter("muF", "lnN")
    #muR = rl.NuisanceParameter("muR", "lnN")
    qcd1 = rl.NuisanceParameter("qcd1", "lnN")
    qcd2 = rl.NuisanceParameter("qcd2", "lnN")
    qcd3 = rl.NuisanceParameter("qcd3", "lnN")
    whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
    zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
    #ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")
    pileup = rl.NuisanceParameter("pileup"+year, "shape")
    
    with open(
        "data/"+"test"+year+".model",
        "wb",) as fout:
        pickle.dump(model(year), fout, protocol=2)

In [39]:
background['ttbarm'].identifiers('systematic')

[<StringBin (btagDown) instance at 0x7fc519a78da0>,
 <StringBin (btagUp) instance at 0x7fc519a78d68>,
 <StringBin (ew1Down) instance at 0x7fc519a736d8>,
 <StringBin (ew1Up) instance at 0x7fc519a73668>,
 <StringBin (ew2GDown) instance at 0x7fc519a73780>,
 <StringBin (ew2GUp) instance at 0x7fc519a73748>,
 <StringBin (ew2WDown) instance at 0x7fc519a73828>,
 <StringBin (ew2WUp) instance at 0x7fc519a737f0>,
 <StringBin (ew2ZDown) instance at 0x7fc519a738d0>,
 <StringBin (ew2ZUp) instance at 0x7fc519a73898>,
 <StringBin (ew3GDown) instance at 0x7fc519a73978>,
 <StringBin (ew3GUp) instance at 0x7fc519a73940>,
 <StringBin (ew3WDown) instance at 0x7fc519a73a20>,
 <StringBin (ew3WUp) instance at 0x7fc519a739e8>,
 <StringBin (ew3ZDown) instance at 0x7fc519a73ac8>,
 <StringBin (ew3ZUp) instance at 0x7fc519a73a90>,
 <StringBin (id_mDown) instance at 0x7fc519a73160>,
 <StringBin (id_mUp) instance at 0x7fc519a73128>,
 <StringBin (isolation_mDown) instance at 0x7fc519a73240>,
 <StringBin (isolation_mU

# wjete

In [ ]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mT_corrected_binning_dict = {
    "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
    "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
}



def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
#     bkg_map["W+HF"] = ("W+HF",)
#     bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z_jets"] = ("Z+*",)
#     bkg_map["Z+HF"] = ("Z+HF",)
#     bkg_map["Z+LF"] = ("Z+LF",)
#     bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SingleMuon"] = ("SingleMuon",)
    data_map["EGamma"] = ("EGamma",)
    sig_map["Mphi1995_Mchi1000"] = ("Mphi-1995_Mchi-1000")
    sig_map["Mphi2000_Mchi1500"] = ("Mphi-2000_Mchi-1500")
    sig_map["Mphi2000_Mchi500"] = ("Mphi-2000_Mchi-500")
#     haven't processed signal yet
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mchi' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial mT_corrected binning',bkg_hists["template"].axis("mT_corrected").edges())


    bkg_hists["template"] = bkg_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "mT_corrected", hist.Bin("mT_corrected", "mT_corrected", mT_corrected_binning)
    )

    hists = {"bkg": bkg_hists, "data": data_hists, "sig":signal_hists}

    return hists


def addBtagSyst(dictionary, process, region, templ):
    btagUp = template(dictionary, process, "btagUp", region)[0]
    btagDown = template(dictionary, process, "btagDown", region)[0]
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, process, region, templ):
    def addSyst(dictionary, process, region, templ, syst, string):
        histogram = dictionary[region].integrate("process", process)
#         print(histogram.integrate("systematic", "nominal").sum('npvs').values()[()])
        nominal=histogram.integrate("systematic", "nominal").sum('npvs').values()[()]
        up=histogram.integrate("systematic", string+"Up").sum('npvs').values()[()]
        down=histogram.integrate("systematic",string+"Down").sum('npvs').values()[()]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, process, region, templ, ew1, "ew1")
    addSyst(dictionary, process, region, templ, ew2W, "ew2W")
    addSyst(dictionary, process, region, templ, ew2Z, "ew2Z")
    addSyst(dictionary, process, region, templ, ew3W, "ew3W")
    addSyst(dictionary, process, region, templ, ew3Z, "ew3Z")
    addSyst(dictionary, process, region, templ, mix, "mix")
    addSyst(dictionary, process, region, templ, qcd1, "qcd1")
    addSyst(dictionary, process, region, templ, qcd2, "qcd2")
    addSyst(dictionary, process, region, templ, qcd3, "qcd3")
    addSyst(dictionary, process, region, templ, muR, "muR")
    addSyst(dictionary, process, region, templ, muF, "muF")
    
def template(dictionary, process, systematic, region, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process).sum('npvs')
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    zerobins = nominal <= 0.
    output = nominal
    
    if "nominal" not in systematic and "data" not in systematic:
    #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("mT_corrected")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "mT_corrected", sumw2)
    return (output, binning, "mT_corrected")

def model(year):

#     model_id = year + category + "recoil" + str(recoil)
#     print(model_id)
#     model = rl.Model(str(s) + model_id)



    model_id = 'w'
    model = rl.Model(model_id)
    ch_name = "wjete" 
    # Create a channel/Control region
    wjete = rl.Channel(ch_name)
    # add the Channel/CR to the model
    model.addChannel(wjete)
    
    ###
    # Add data distribution to the channel
    ###

    if year == "2018":
        wjete.setObservation(template(data, "EGamma", "data", "wjete"))
    else:
        wjete.setObservation(template(data, "SingleElectron", "data","wjete"))
        
    #template for signal
    for s in signal['wjete'].identifiers('process'):
        print(s)
        if 'Mphi1995' not in str(s): continue
        wjete_signalTemplate = template(signal,s,'nominal','wjete')
        wjete_signal=rl.TemplateSample(ch_name+'_'+str(s), rl.Sample.SIGNAL, wjete_signalTemplate)
        wjete_signal.setParamEffect(lumi, 1.027)
    #     sr_signal.setParamEffect(trig_met, 1.01)
    #     sr_signal.setParamEffect(veto_tau, 1.03)
    #     sr_signal.setParamEffect(jec, 1.05)
        #btag
        btagUp=template(signal, s,'btagUp','wjete')[0]
        btagDown=template(signal, s,'btagDown','wjete')[0]
        wjete_signal.setParamEffect(btag, btagUp, btagDown)
        #trigger Electron
        trigeUp=template(signal, s, 'trig_eUp', 'wjete')[0] 
        trigeDown=template(signal, s, 'trig_eDown', 'wjete')[0]
        wjete_signal.setParamEffect(trig_e, trigeUp, trigeDown)
        #id Electron
        id_eUp=template(signal, s, 'id_eUp', 'wjete')[0] 
        id_eDown=template(signal, s, 'id_eDown', 'wjete')[0]
        wjete_signal.setParamEffect(id_e, id_eUp, id_eDown)
        #pileup
        pileupUp=template(signal, s,'pileupUp','wjete')[0] 
        pileupDown=template(signal, s,'pileupDown','wjete')[0] 
        wjete_signal.setParamEffect(pileup, pileupUp, pileupDown)
        
        #reco Electron
        recoeUp=template(signal, s, 'reco_eUp', 'wjete')[0] 
        recoDown=template(signal, s, 'reco_eDown', 'wjete')[0]
        wjete_signal.setParamEffect(trig_e, recoeUp, recoDown)
        
        wjete.addSample(wjete_signal)
    ###
    # W(->lnu)+jets data-driven model
    ###

    wjete_wjetsTemplate = template(background, "W+jets", "nominal", "wjete", read_sumw2=True)
    wjete_wjetsMC = rl.TemplateSample(
        ch_name + "_wjetsMC",
        rl.Sample.BACKGROUND,
        wjete_wjetsTemplate
    )
    wjete_wjetsMC.setParamEffect(lumi, 1.027)
#     wjete_wjetsMC.setParamEffect(trig_e, 1.01)
#     wjete_wjetsMC.setParamEffect(veto_tau, 1.03)
#     wjete_wjetsMC.setParamEffect(wjets_norm, 1.4)
#     wjete_wjetsMC.setParamEffect(jec, 1.05)
#     wjete_wjetsMC.setParamEffect(id_e, 1.02)
#     wjete_wjetsMC.setParamEffect(reco_e, 1.02)
    #wjete_wjetsMC.autoMCStats()
    addBtagSyst(background, "W+jets", "wjete", wjete_wjetsMC)
    #trigger Electron
    trigeUp=template(background, 'W+jets', 'trig_eUp', 'wjete')[0] 
    trigeDown=template(background, 'W+jets', 'trig_eDown', 'wjete')[0]
    wjete_wjetsMC.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "W+jets", 'id_eUp', 'wjete')[0] 
    id_eDown=template(background, "W+jets", 'id_eDown', 'wjete')[0]
    wjete_wjetsMC.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'W+jets','pileupUp','wjete')[0] 
    pileupDown=template(background, 'W+jets','pileupDown','wjete')[0] 
    wjete_wjetsMC.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'W+jets', 'reco_eUp', 'wjete')[0] 
    recoDown=template(background, 'W+jets', 'reco_eDown', 'wjete')[0]
    wjete_wjetsMC.setParamEffect(trig_e, recoeUp, recoDown)
    # addVJetsSyst(background, "W+jets", "wjete", wjete_wjetsMC)
    wjete.addSample(wjete_wjetsMC)

    # ###
    # # top-antitop model
    # ###

    wjete_ttTemplate = template(background, "TT", "nominal","wjete", read_sumw2=True)
    wjete_ttMC = rl.TemplateSample(
        ch_name + "_ttMC",
        rl.Sample.BACKGROUND,
        wjete_ttTemplate
    )
    wjete_ttMC.setParamEffect(lumi, 1.027)
#     wjete_ttMC.setParamEffect(trig_e, 1.01)
#     wjete_ttMC.setParamEffect(veto_tau, 1.03)
#     wjete_ttMC.setParamEffect(jec, 1.05)
#     wjete_ttMC.setParamEffect(id_e, 1.02)
#     wjete_ttMC.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "TT", "wjete", wjete_ttMC)
    #trigger Electron
    trigeUp=template(background, 'TT', 'trig_eUp', 'wjete')[0] 
    trigeDown=template(background, 'TT', 'trig_eDown', 'wjete')[0]
    wjete_ttMC.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "TT", 'id_eUp', 'wjete')[0] 
    id_eDown=template(background, "TT", 'id_eDown', 'wjete')[0]
    wjete_ttMC.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'TT','pileupUp','wjete')[0] 
    pileupDown=template(background, 'TT','pileupDown','wjete')[0] 
    wjete_ttMC.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'TT', 'reco_eUp', 'wjete')[0] 
    recoDown=template(background, 'TT', 'reco_eDown', 'wjete')[0]
    wjete_ttMC.setParamEffect(trig_e, recoeUp, recoDown)
    # wjete_ttMC.setParamEffect(ttMC_norm, 1.2)
    wjete.addSample(wjete_ttMC)

    # ###
    # # Other MC-driven processes
    # ###
    # ST Background
    wjete_stTemplate = template(background, "ST", "nominal", "wjete")
    wjete_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, wjete_stTemplate
    )
    wjete_st.setParamEffect(lumi, 1.027)
#     wjete_st.setParamEffect(trig_e, 1.01)
#     wjete_st.setParamEffect(veto_tau, 1.03)
#     wjete_st.setParamEffect(st_norm, 1.2) something related to signal
#     wjete_st.setParamEffect(jec, 1.05)
#     wjete_st.setParamEffect(id_e, 1.02)
#     wjete_st.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "ST", "wjete", wjete_st)
    #trigger Electron
    trigeUp=template(background, 'ST', 'trig_eUp', 'wjete')[0] 
    trigeDown=template(background, 'ST', 'trig_eDown', 'wjete')[0]
    wjete_st.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "ST", 'id_eUp', 'wjete')[0] 
    id_eDown=template(background, "ST", 'id_eDown', 'wjete')[0]
    wjete_st.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'ST','pileupUp','wjete')[0] 
    pileupDown=template(background, 'ST','pileupDown','wjete')[0] 
    wjete_st.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'ST', 'reco_eUp', 'wjete')[0] 
    recoDown=template(background, 'ST', 'reco_eDown', 'wjete')[0]
    wjete_st.setParamEffect(trig_e, recoeUp, recoDown)
    wjete.addSample(wjete_st)

    # DY Jet Background
    wjete_dyjetsTemplate = template(background, "DY+jets", "nominal","wjete")
    wjete_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, wjete_dyjetsTemplate
    )
    wjete_dyjets.setParamEffect(lumi, 1.027)
#     wjete_dyjets.setParamEffect(trig_e, 1.01)
#     wjete_dyjets.setParamEffect(veto_tau, 1.03)
#     wjete_dyjets.setParamEffect(zjetsMC_norm, 1.4)
#     wjete_dyjets.setParamEffect(jec, 1.05)
#     wjete_dyjets.setParamEffect(id_e, 1.02)
#     wjete_dyjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "DY+jets", "wjete", wjete_dyjets)
    addVJetsSyst(background,"DY+jets", "wjete", wjete_dyjets)
    #trigger Electron
    trigeUp=template(background, 'DY+jets', 'trig_eUp', 'wjete')[0] 
    trigeDown=template(background, 'DY+jets', 'trig_eDown', 'wjete')[0]
    wjete_dyjets.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "DY+jets", 'id_eUp', 'wjete')[0] 
    id_eDown=template(background, "DY+jets", 'id_eDown', 'wjete')[0]
    wjete_dyjets.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'DY+jets','pileupUp','wjete')[0] 
    pileupDown=template(background, 'DY+jets','pileupDown','wjete')[0] 
    wjete_dyjets.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'DY+jets', 'reco_eUp', 'wjete')[0] 
    recoDown=template(background, 'DY+jets', 'reco_eDown', 'wjete')[0]
    wjete_dyjets.setParamEffect(trig_e, recoeUp, recoDown)
    wjete.addSample(wjete_dyjets)


    # V Jets Background
    wjete_vvTemplate = template(background, "VV", "nominal", "wjete")
    wjete_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, wjete_vvTemplate
    )
    wjete_vv.setParamEffect(lumi, 1.027)
#     wjete_vv.setParamEffect(trig_e, 1.01)
#     wjete_vv.setParamEffect(veto_tau, 1.03)
#     wjete_vv.setParamEffect(vv_norm, 1.2)
#     wjete_vv.setParamEffect(jec, 1.05)
#     wjete_vv.setParamEffect(id_e, 1.02)
#     wjete_vv.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "VV", "wjete", wjete_vv)
    #trigger Electron
    trigeUp=template(background, 'VV', 'trig_eUp', 'wjete')[0] 
    trigeDown=template(background, 'VV', 'trig_eDown', 'wjete')[0]
    wjete_vv.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "VV", 'id_eUp', 'wjete')[0] 
    id_eDown=template(background, "VV", 'id_eDown', 'wjete')[0]
    wjete_vv.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'VV','pileupUp','wjete')[0] 
    pileupDown=template(background, 'VV','pileupDown','wjete')[0] 
    wjete_vv.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'VV', 'reco_eUp', 'wjete')[0] 
    recoDown=template(background, 'VV', 'reco_eDown', 'wjete')[0]
    wjete_vv.setParamEffect(trig_e, recoeUp, recoDown)
    
    wjete.addSample(wjete_vv)

    # QCD background
    wjete_qcdTemplate = template(background, "QCD", "nominal", "wjete")
    wjete_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, wjete_qcdTemplate
    )
    wjete_qcd.setParamEffect(lumi, 1.027)
#     wjete_qcd.setParamEffect(trig_e, 1.01)
#     wjete_qcd.setParamEffect(veto_tau, 1.03)
#     wjete_qcd.setParamEffect(qcde_norm, 2.0)
#     wjete_qcd.setParamEffect(jec, 1.05)
#     wjete_qcd.setParamEffect(id_e, 1.02)
#     wjete_qcd.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, "QCD", "wjete", wjete_qcd)
    #trigger Electron
    trigeUp=template(background, 'QCD', 'trig_eUp', 'wjete')[0] 
    trigeDown=template(background, 'QCD', 'trig_eDown', 'wjete')[0]
    wjete_qcd.setParamEffect(trig_e, trigeUp, trigeDown)
    #id Electron
    id_eUp=template(background, "QCD", 'id_eUp', 'wjete')[0] 
    id_eDown=template(background, "QCD", 'id_eDown', 'wjete')[0]
    wjete_qcd.setParamEffect(id_e, id_eUp, id_eDown)
    #pileup
    pileupUp=template(background, 'QCD','pileupUp','wjete')[0] 
    pileupDown=template(background, 'QCD','pileupDown','wjete')[0] 
    wjete_qcd.setParamEffect(pileup, pileupUp, pileupDown)
    #reco Electron
    recoeUp=template(background, 'QCD', 'reco_eUp', 'wjete')[0] 
    recoDown=template(background, 'QCD', 'reco_eDown', 'wjete')[0]
    wjete_qcd.setParamEffect(trig_e, recoeUp, recoDown)
    
    wjete.addSample(wjete_qcd)
    return model

if __name__ == "__main__":
    if not os.path.exists("datacards"):
        os.mkdir("datacards")
    parser = OptionParser()
    parser.add_option("-y", "--year", help="year", dest="year", default="")
    (options, args) = parser.parse_args()
    year = options.year
    year = '2018'

    mT_corrected_binning_dict = {
        "2018": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2017": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500]),
        "2016": np.array([100.0,125.0,150.0, 165.0, 180.0, 200.0, 220.0, 240.0, 270.0, 310.0, 350.0, 500])
    }
    mT_corrected_binning = mT_corrected_binning_dict[year]
    mT_correctedbins = np.array(mT_corrected_binning)
    nmT_corrected = len(mT_correctedbins) - 1

    ###
    # Extract histograms from input file
    ###

    print("Grouping histograms")
    hists = load("hists/UL_lep_monotop_v1_test_" + year + ".scaled")
    hists = remap_histograms(hists)
    data_hists = hists["data"]
    bkg_hists = hists["bkg"]
    signal_hists = hists["sig"]

    ###
    # Preparing histograms for fit
    ##

    data = {}
    for r in data_hists["template"].identifiers("region"):
        data[str(r)] = data_hists["template"].integrate("region", r)

    background = {}
    for r in bkg_hists["template"].identifiers("region"):
        background[str(r)] = bkg_hists["template"].integrate("region", r)

    signal = {}
    for r in signal_hists["template"].identifiers("region"):
        signal[str(r)] = signal_hists["template"].integrate("region", r)
    ###
    ###
    # Setting up systematics
    ###
    ###
    lumi = rl.NuisanceParameter("lumi" + year, "lnN")
    zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
    wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
    tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
    id_e = rl.NuisanceParameter("id_e" + year, "shape")
    id_mu = rl.NuisanceParameter("id_mu" + year, "lnN")
    id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
    reco_e = rl.NuisanceParameter("reco_e" + year, "shape")
    iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
    trig_e = rl.NuisanceParameter("trig_e" + year, "shape")
    trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
    trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
    veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
    jec = rl.NuisanceParameter("jec" + year, "lnN")
    #btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
    btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
    ew1 = rl.NuisanceParameter("ew1", "lnN")
    #ew2G = rl.NuisanceParameter("ew2G", "lnN")
    ew2W = rl.NuisanceParameter("ew2W", "lnN")
    ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
    #ew3G = rl.NuisanceParameter("ew3G", "lnN")
    ew3W = rl.NuisanceParameter("ew3W", "lnN")
    ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
    mix = rl.NuisanceParameter("mix", "lnN")
    muF = rl.NuisanceParameter("muF", "lnN")
    muR = rl.NuisanceParameter("muR", "lnN")
    qcd1 = rl.NuisanceParameter("qcd1", "lnN")
    qcd2 = rl.NuisanceParameter("qcd2", "lnN")
    qcd3 = rl.NuisanceParameter("qcd3", "lnN")
    whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
    zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
    #ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")
    pileup = rl.NuisanceParameter("pileup"+year, "shape")
    
    with open(
        "data/"+"test_wjete"+year+".model",
        "wb",) as fout:
        pickle.dump(model(year), fout, protocol=2)

In [17]:
str_ = "MCHI_"
if 'MCHI' in str_:
    print("hi")

hi


In [23]:
for s in signal['wjete'].identifiers('process'):
    print(s)

Mphi1995_Mchi1000
Mphi2000_Mchi1500
Mphi2000_Mchi500


In [24]:
for s in signal['ttbare'].identifiers('process'):
    print(s)

Mphi1995_Mchi1000
Mphi2000_Mchi1500
Mphi2000_Mchi500
